# Lean-19 : Detection MIMO par index de comptage -- banc d'essai Hoeffding

Carnet de **banc d'essai methodologique** pour la serie MIMO (sous-serie [MIMO-01..04](MIMO/README.md)). L'invariant postule est trivial -- la loi de Bernoulli composee, `p_hat ~ p` -- mais sa verification empirique sur 20 instances aleatoires etablit un **etalon de mesure** pour les carnets MIMO de la sous-serie.

**Navigation** : [Version Python](Lean-19-Detection-MIMO-Index-Python.ipynb) · [<< Lean-17 Pont Conway](Lean-17-Conway-Bridge-to-Knots-Python.ipynb) · [Lean-19b Chernoff](Lean-19b-Borne-Chernoff-Python.ipynb) · [Lean-19c Bernstein](Lean-19c-Borne-Bernstein-CSharp.ipynb) · [Lean-19d Bayésien](Lean-19d-Inference-Bayesian-Beta-Binomial-Python.ipynb) · [Sommaire série](README.md)

**Enonce** : Soit un canal binaire symetrique (BSC) a `M` etats, observe sur `t` pas de temps, avec probabilite de flip `p` par pas. Chaque indicatrice de flip suit une loi de Bernoulli de parametre `p` ; le nombre total de flips sur `N = n_runs * t` tirages independants suit une loi binomiale. La borne de Hoeffding donne le seuil suivant :

```
|p_hat - p| <= sqrt(log(2/eps) / (2 * N))
```

L'invariant tient ssi `|p_hat - p|` reste sous la borne pour `eps = 0.05` (5 % de risque de dépassement au plus).

**Pourquoi ce carnet n'est pas dans la sous-serie MIMO** : la sous-serie MIMO-01..04 vit dans [MIMO/](MIMO/README.md). Lean-19, dans la serie principale, fournit la **verification independante** que l'invariant de comptage tient avant la capstone. Lean-19 y sert d'etalon de reference pour la detection MIMO.

Protocole de mesure : pour chaque instance, le canal est simulé `n_runs` fois sur `t` pas — à chaque pas, un flip a lieu avec probabilité `p` (`rng.NextDouble() < p`) et l'état avance modulo `M`. On obtient `N = n_runs * t` tirages, la fréquence observée `p_hat = flips / N`, et l'invariant `|p_hat - p| <= borne` est testé ligne par ligne dans les tableaux imprimés.

La borne de Hoeffding majore une probabilité : avec `eps = 0.05`, l'événement « l'écart dépasse la borne » a une probabilité d'au plus 5 %. C'est une garantie sur la queue de distribution, pas une valeur typique de l'écart.

**Note de parité** : les entrées et les tableaux de bornes sont identiques à ceux du carnet Python, à la précision affichée. Les statistiques de tirage sont recalculées avec `System.Random` et la graine 42 ; elles varient d'un générateur à l'autre. Les écarts Python utilisés pour la comparaison sont des références citées, pas des résultats du port. Les invariants `12/12`, `8/8` et `20/20` sous la borne sont vérifiés par des assertions.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Graine pour reproductibilite
var graine = 42;
var rng = new Random(graine);

// Trois erreurs-types : repère Monte-Carlo par instance, pas un seuil simultané.
const double MultiplicateurMonteCarlo = 3.0;
// Demi-unité du dernier chiffre des écarts cités, imprimés à quatre décimales.
const double ArrondiCitationF4 = 0.00005;

string Inv(FormattableString texte) => texte.ToString(CultureInfo.InvariantCulture);

void Exiger(bool condition, string message)
{
    if (!condition)
        throw new InvalidOperationException(message);
}

// Les références citées et les mesures .NET restent dans des champs distincts.
record ReferencePython(int M, int T, double P, int N, string Borne, double Ecart);
record Mesure(int M, int T, double P, int N, int Flips, double PHat, double Ecart,
    double Borne, bool OK, double SeuilMonteCarlo, double EcartPython,
    double DeltaPython, double SeuilPython, double DeltaBorneAffichee);

Exiger(graine == 42, "La graine doit être celle du protocole : 42.");
Console.WriteLine(Inv($"Lean-19 seed={graine} (rng .NET System.Random)"));

Lean-19 seed=42 (rng .NET System.Random)


### Lecture de la sortie : la graine est fixée

La sortie affiche `Lean-19 seed=42 (rng .NET System.Random)` : le générateur pseudo-aléatoire est ancré sur la graine 42, donc réexécuter le carnet dans le même environnement .NET rejoue exactement les mêmes tirages et produit les mêmes tableaux. `System.Random` ne reproduit pas la suite de `PCG64` utilisée par NumPy.

L'erreur-type de `p_hat` est `sqrt(p(1-p)/N)`, car le nombre de flips est binomial. Le contrôle Monte-Carlo utilise trois erreurs-types par instance : c'est un repère de dispersion, pas une garantie simultanée sur les 20 instances. Pour comparer les écarts C# et Python, on utilise trois erreurs-types de la différence de deux fréquences indépendantes, soit `3 * sqrt(2p(1-p)/N)`, plus le demi-pas d'arrondi de la valeur Python citée. Ce seuil s'applique aussi à la différence des écarts absolus, car prendre la valeur absolue ne peut pas agrandir cette différence.

In [2]:
double HoeffdingBorne(int N, double eps = 0.05) =>
    Math.Sqrt(Math.Log(2.0 / eps) / (2.0 * N));

// 12 instances de reference : BSC a M etats, t pas, probabilite p de flip
// Références CITÉES : entrées, borne affichée et écart affiché, sortie Python de la cellule 3.
var references12 = new ReferencePython[]
{
    new(4,  3, 0.100, 3000, "0.0248", 0.0007),
    new(5,  4, 0.150, 4000, "0.0215", 0.0040),
    new(6,  5, 0.200, 5000, "0.0192", 0.0032),
    new(7,  3, 0.250, 3000, "0.0248", 0.0077),
    new(8,  4, 0.300, 4000, "0.0215", 0.0055),
    new(9,  5, 0.350, 5000, "0.0192", 0.0012),
    new(10, 3, 0.400, 3000, "0.0248", 0.0143),
    new(11, 4, 0.450, 4000, "0.0215", 0.0023),
    new(12, 5, 0.500, 5000, "0.0192", 0.0134),
    new(13, 3, 0.550, 3000, "0.0248", 0.0090),
    new(14, 4, 0.600, 4000, "0.0215", 0.0068),
    new(15, 5, 0.650, 5000, "0.0192", 0.0010)
};

Mesure Simuler(int m, int t, double p, int nRuns, ReferencePython citation)
{
    int n = nRuns * t;
    Exiger(m == citation.M && t == citation.T && n == citation.N &&
        p.ToString("F3", CultureInfo.InvariantCulture) == citation.P.ToString("F3", CultureInfo.InvariantCulture),
        Inv($"M={m}, t={t}, p={p:F3}, N={n} : entrées différentes de la référence Python citée."));

    double borne = HoeffdingBorne(n);
    string borneAffichee = borne.ToString("F4", CultureInfo.InvariantCulture);
    Exiger(borneAffichee == citation.Borne,
        Inv($"M={m}, N={n} : borne .NET={borneAffichee}, borne Python citée={citation.Borne}."));
    double deltaBorneAffichee = Math.Abs(double.Parse(borneAffichee, CultureInfo.InvariantCulture)
        - double.Parse(citation.Borne, CultureInfo.InvariantCulture));

    // Fixer les seuils à partir de p et N, avant de tirer les états et les flips.
    double varianceFrequence = p * (1.0 - p) / n;
    double seuilMonteCarlo = MultiplicateurMonteCarlo * Math.Sqrt(varianceFrequence);
    double seuilPython = MultiplicateurMonteCarlo * Math.Sqrt(2.0 * varianceFrequence) + ArrondiCitationF4;
    int flips = 0;
    for (int passe = 0; passe < nRuns; passe++)
    {
        int etat = rng.Next(m);
        for (int pas = 0; pas < t; pas++)
        {
            if (rng.NextDouble() < p)
            {
                etat = (etat + 1) % m;
                flips++;
            }
        }
    }

    double pHat = (double)flips / n;
    double ecart = Math.Abs(pHat - p);
    double deltaPython = Math.Abs(ecart - citation.Ecart);
    Exiger(flips >= 0 && flips <= n, Inv($"M={m} : inversions hors de [0, {n}]."));
    Exiger(ecart <= seuilMonteCarlo,
        Inv($"M={m}, p={p:F3} : |p_hat-p|={ecart:G17} dépasse 3 erreurs-types ({seuilMonteCarlo:G17})."));
    Exiger(deltaPython <= seuilPython,
        Inv($"M={m}, p={p:F3} : différence avec l'écart Python cité={deltaPython:G17}, seuil={seuilPython:G17}."));
    return new Mesure(m, t, p, n, flips, pHat, ecart, borne, ecart <= borne,
        seuilMonteCarlo, citation.Ecart, deltaPython, seuilPython, deltaBorneAffichee);
}

void AfficherLot(IReadOnlyList<Mesure> mesures, int attendu, int nRuns, string titre)
{
    Exiger(mesures.Count == attendu && mesures.All(x => x.N / x.T == nRuns),
        "L'effectif ou le nombre de passes diffère du protocole Python.");
    int sousBorne = mesures.Count(x => x.OK);
    double pourcentage = 100.0 * sousBorne / mesures.Count;
    Exiger(sousBorne == attendu && pourcentage == 100.0,
        Inv($"Invariant Hoeffding : {sousBorne}/{mesures.Count} sous la borne, attendu {attendu}/{attendu}."));

    Console.WriteLine(titre);
    Console.WriteLine($"{"M",4} {"t",3} {"p",6} {"N",6} {"flips",6} {"p_hat",7} {"ecart",7} {"borne",7} {"OK",4}");
    foreach (var x in mesures)
        Console.WriteLine(Inv($"{x.M,4} {x.T,3} {x.P,6:F3} {x.N,6} {x.Flips,6} {x.PHat,7:F4} {x.Ecart,7:F4} {x.Borne,7:F4} {(x.OK ? "OUI" : "NON"),4}"));

    Console.WriteLine();
    Console.WriteLine("Contrôles Monte-Carlo (.NET et écarts Python cités) :");
    foreach (var x in mesures)
        Console.WriteLine(Inv($"M={x.M,2}, p={x.P:F3} : écart .NET={x.Ecart:F6} (seuil={x.SeuilMonteCarlo:F6}) ; écart Python cité={x.EcartPython:F4} ; différence={x.DeltaPython:F6} (seuil={x.SeuilPython:F6}) — CONFORME"));

    var pire = mesures.OrderByDescending(x => x.Ecart).First();
    Console.WriteLine(Inv($"Écart maximal .NET : {pire.Ecart:F4} (M={pire.M}, p={pire.P:F3}, borne={pire.Borne:F4})."));
    Console.WriteLine(Inv($"--> {sousBorne}/{mesures.Count} instances sous la borne Hoeffding ({pourcentage:F0} %)"));
}

const int passesRef = 1000;
Exiger(references12.Length == 12, "La référence Python doit contenir les 12 instances.");
var instancesRef = new List<Mesure>();
for (int i = 0; i < 12; i++)
    instancesRef.Add(Simuler(4 + i, 3 + i % 3, 0.10 + 0.05 * i, passesRef, references12[i]));

AfficherLot(instancesRef, 12, passesRef,
    Inv($"=== Cellule 1 : {instancesRef.Count} instances de reference (Monte-Carlo, n_runs={passesRef}) ==="));

=== Cellule 1 : 12 instances de reference (Monte-Carlo, n_runs=1000) ===


   M   t      p      N  flips   p_hat   ecart   borne   OK


   4   3  0.100   3000    327  0.1090  0.0090  0.0248  OUI


   5   4  0.150   4000    610  0.1525  0.0025  0.0215  OUI


   6   5  0.200   5000   1029  0.2058  0.0058  0.0192  OUI


   7   3  0.250   3000    776  0.2587  0.0087  0.0248  OUI


... (19 outputs left out)

M=14, p=0.600 : écart .NET=0.010250 (seuil=0.023238) ; écart Python cité=0.0068 ; différence=0.003450 (seuil=0.032913) — CONFORME


M=15, p=0.650 : écart .NET=0.007600 (seuil=0.020236) ; écart Python cité=0.0010 ; différence=0.006600 (seuil=0.028668) — CONFORME


Écart maximal .NET : 0.0108 (M=8, p=0.300, borne=0.0215).


--> 12/12 instances sous la borne Hoeffding (100 %)


### Lecture du résultat : 12/12 instances sous la borne

La sortie se conclut par `--> 12/12 instances sous la borne Hoeffding (100 %)` et chaque ligne affiche `OUI` dans la colonne `OK`. Pour `M=4, p=0.100`, la fréquence observée est `p_hat=0.1090` : l'écart de `0.0090` reste sous la borne de `0.0248`. Le plus grand écart du lot .NET, `0.0108`, est observé pour `M=8, p=0.300` et reste lui aussi sous sa borne de `0.0215`.

## Mesure 2 : extension sur 8 nouvelles instances

Les 12 instances precedentes varient `M` (4 a 15) et `t` (3 a 5) avec `p` lineaire de 0.10 a 0.65. Pour verifier que l'invariant tient au-dela de cette plage, on elargit a 8 nouvelles instances avec `M` (6 a 20) et `p` plus eleve (0.20 a 0.76) et `n_runs` double (2000), ce qui reduit la borne Hoeffding d'un facteur `sqrt(2)` à `t` constant.

Rappel de la mécanique mesurée : `p_hat = flips / N` est la fréquence observée des flips, et la colonne `OK` vaut `OUI` dès que `|p_hat - p| <= sqrt(log(2/eps) / (2N))` avec `eps = 0.05`. Doubler `n_runs` divise la borne par `sqrt(2)` (environ 0,707) : c'est pourquoi les bornes affichées dans cette cellule (0,0136 à 0,0152) sont plus serrées que celles de la cellule 1 (0,0192 à 0,0248).

Élargir la plage a un sens pour la suite : la détection MIMO opere sur des canaux à plus grand nombre d'états que les petites valeurs de `M` de la cellule 1, donc l'invariant doit tenir au-delà de cette plage avant d'être invoqué dans la capstone.

In [3]:
// Illustration 2 — Extension sur 8 nouvelles instances
// 8 nouvelles instances : plage elargie (M, p) et n_runs double
// Références CITÉES : entrées, borne affichée et écart affiché, sortie Python de la cellule 6.
var references8 = new ReferencePython[]
{
    new(6,  4, 0.200,  8000, "0.0152", 0.0003),
    new(8,  5, 0.280, 10000, "0.0136", 0.0026),
    new(10, 4, 0.360,  8000, "0.0152", 0.0055),
    new(12, 5, 0.440, 10000, "0.0136", 0.0004),
    new(14, 4, 0.520,  8000, "0.0152", 0.0096),
    new(16, 5, 0.600, 10000, "0.0136", 0.0074),
    new(18, 4, 0.680,  8000, "0.0152", 0.0015),
    new(20, 5, 0.760, 10000, "0.0136", 0.0041)
};

const int passesExtension = 2000;
Exiger(references8.Length == 8 && passesExtension == 2 * passesRef,
    "L'extension doit contenir les 8 instances et doubler le nombre de passes.");

// Poursuivre le même flux .NET : aucune réinitialisation de rng entre les lots.
var instancesNew = new List<Mesure>();
for (int i = 0; i < 8; i++)
    instancesNew.Add(Simuler(6 + 2 * i, 4 + i % 2, 0.20 + 0.08 * i, passesExtension, references8[i]));

AfficherLot(instancesNew, 8, passesExtension,
    Inv($"=== Cellule 2 : {instancesNew.Count} nouvelles instances (extension, n_runs={passesExtension}) ==="));

=== Cellule 2 : 8 nouvelles instances (extension, n_runs=2000) ===


   M   t      p      N  flips   p_hat   ecart   borne   OK


   6   4  0.200   8000   1633  0.2041  0.0041  0.0152  OUI


   8   5  0.280  10000   2863  0.2863  0.0063  0.0136  OUI


  10   4  0.360   8000   2873  0.3591  0.0009  0.0152  OUI


  12   5  0.440  10000   4485  0.4485  0.0085  0.0136  OUI


... (11 outputs left out)

M=18, p=0.680 : écart .NET=0.000250 (seuil=0.015646) ; écart Python cité=0.0015 ; différence=0.001250 (seuil=0.022177) — CONFORME


M=20, p=0.760 : écart .NET=0.001200 (seuil=0.012812) ; écart Python cité=0.0041 ; différence=0.002900 (seuil=0.018170) — CONFORME


Écart maximal .NET : 0.0085 (M=12, p=0.440, borne=0.0136).


--> 8/8 instances sous la borne Hoeffding (100 %)


### Lecture du résultat : l'extension tient aussi

La sortie affiche `--> 8/8 instances sous la borne Hoeffding (100 %)` sur la plage élargie (`M` de 6 à 20, `p` de 0.20 à 0.76). Les bornes y vont de `0.0136` à `0.0152`. À `t` constant, doubler `n_runs` divise la borne par `sqrt(2)`, mais l'extension change aussi `t` (`t = 4 + (i % 2)`), et `N` atteint `10000`. Le passage de `0.0248` à `0.0152` combine les deux effets (`N` passe de `3000` à `8000`, soit un facteur de réduction de `sqrt(8000/3000)`, environ `1.63`). Le plus grand écart observé dans l'extension .NET est `0.0085` pour `M=12, p=0.440`, sous la borne de `0.0136`.

## Mesure 3 : confrontation Hoeffding globale (20 instances)

Agregation des 20 instances : ecart observe `|p_hat - p|`, rapport a la borne Hoeffding `|p_hat - p| / borne`. Si le rapport reste inferieur a 1 sur toutes les instances, l'invariant tient avec une marge indiquee par le rapport max (le pire cas).

Le rapport `ecart / borne` est sans unité : il vaut 1 quand la borne est atteinte exactement, et moins de 1 quand l'invariant tient avec de la marge. Son `max` sur les 20 instances est donc le pire cas observé — celui qui décide si l'invariant tient — et son `mean` mesure la marge typique.

La cellule 3 imprimera trois lignes de synthèse — écarts min/max/mean, rapports min/max/mean, puis le compte final — à lire comme le verdict global de la mesure sur les 20 instances.

Les valeurs de synthèse Python sont citées pour comparaison, pas réutilisées comme résultats .NET. Pour un minimum ou un maximum, on propage le plus grand seuil par instance ; pour une moyenne, on propage la moyenne des seuils. Les seuils des rapports s'obtiennent en divisant les seuils des écarts par les bornes non arrondies. On ajoute ensuite le demi-pas d'arrondi de chaque statistique citée.

In [4]:
// Illustration 3 — Confrontation Hoeffding globale
// Références CITÉES : synthèse affichée dans la cellule 9 du carnet Python.
var syntheseEcartsPython = (Min: 0.0003, Max: 0.0143, Moyenne: 0.0050);
var syntheseRapportsPython = (Min: 0.016, Max: 0.698, Moyenne: 0.265);
const int effectifPython = 20;
const int sousBornePython = 20;
const double pourcentageSousBornePython = 100.0;
const double pourcentagePirePython = 70.0;

// Demi-pas d'arrondi des citations ; la précision F4 est déclarée au préambule.
const double ArrondiCitationF3 = 0.0005;
const double ArrondiCitationF0 = 0.5;
var allInst = instancesRef.Concat(instancesNew).ToArray();
Exiger(allInst.Length == effectifPython,
    Inv($"Effectif global .NET={allInst.Length}, attendu Python cité={effectifPython}."));

// Le min et le max varient d'au plus le plus grand seuil par ligne ;
// l'inégalité triangulaire donne le seuil de la moyenne.
var seuilsEcarts = allInst.Select(x => x.SeuilPython).ToArray();
var seuilsRapports = allInst.Select(x => x.SeuilPython / x.Borne).ToArray();
double seuilExtremumEcart = seuilsEcarts.Max() + ArrondiCitationF4;
double seuilMoyenneEcart = seuilsEcarts.Average() + ArrondiCitationF4;
double seuilExtremumRapport = seuilsRapports.Max() + ArrondiCitationF3;
double seuilMoyenneRapport = seuilsRapports.Average() + ArrondiCitationF3;
double seuilPourcentagePire = 100.0 * seuilsRapports.Max() + ArrondiCitationF0;

var ecarts = allInst.Select(x => x.Ecart).ToArray();
var rapports = allInst.Select(x => x.Ecart / x.Borne).ToArray();
var bilanEcarts = (Min: ecarts.Min(), Max: ecarts.Max(), Moyenne: ecarts.Average());
var bilanRapports = (Min: rapports.Min(), Max: rapports.Max(), Moyenne: rapports.Average());
var pireRapport = allInst[Array.IndexOf(rapports, bilanRapports.Max)];
double pourcentagePire = 100.0 * bilanRapports.Max;
int nOkGlobal = allInst.Count(x => x.OK);
double pourcentageGlobal = 100.0 * nOkGlobal / allInst.Length;
Exiger(nOkGlobal == sousBornePython && pourcentageGlobal == pourcentageSousBornePython && bilanRapports.Max < 1.0,
    Inv($"Invariant global : .NET={nOkGlobal}/{allInst.Length} ({pourcentageGlobal:F0} %), Python cité={sousBornePython}/{effectifPython} ({pourcentageSousBornePython:F0} %), rapport maximal .NET={bilanRapports.Max:G17}."));

Console.WriteLine(Inv($"=== Cellule 3 : Confrontation Hoeffding sur {allInst.Length} instances ==="));
Console.WriteLine(Inv($"Ecarts observes : min={bilanEcarts.Min:F4}, max={bilanEcarts.Max:F4}, mean={bilanEcarts.Moyenne:F4}"));
Console.WriteLine(Inv($"Rapport ecart/borne : min={bilanRapports.Min:F3}, max={bilanRapports.Max:F3}, mean={bilanRapports.Moyenne:F3}"));
Console.WriteLine(Inv($"--> {nOkGlobal}/{allInst.Length} instances sous la borne Hoeffding ({pourcentageGlobal:F0} %)"));
Console.WriteLine(Inv($"Marge : le pire ecart atteint {pourcentagePire:F0} % de la borne."));
Console.WriteLine(Inv($"Pire rapport .NET : M={pireRapport.M}, t={pireRapport.T}, p={pireRapport.P:F3}, écart={pireRapport.Ecart:F4}, borne={pireRapport.Borne:F4}."));

// Attestation : chaque lot compare les bornes affichées et les écarts observés.
void AttesterLot(string nom, IReadOnlyList<Mesure> lot)
{
    double deltaBorne = lot.Max(x => x.DeltaBorneAffichee);
    double deltaEcart = lot.Max(x => x.DeltaPython);
    double fractionSeuil = lot.Max(x => x.DeltaPython / x.SeuilPython);
    int conformes = lot.Count(x => x.DeltaBorneAffichee == 0.0 &&
        x.DeltaPython <= x.SeuilPython && x.Ecart <= x.SeuilMonteCarlo && x.OK);
    Exiger(conformes == lot.Count,
        Inv($"{nom} : {conformes}/{lot.Count} instances conformes aux contrôles de parité."));
    Console.WriteLine(Inv($"{nom} : Δ max borne affichée={deltaBorne:F4} ; Δ max écart (.NET/Python cité)={deltaEcart:F6} ; part maximale du seuil par instance={fractionSeuil:P1} ; {conformes}/{lot.Count} — CONFORME"));
}

void ComparerSynthese(string nom, double mesure, double citation, double seuil, string format)
{
    double delta = Math.Abs(mesure - citation);
    string comparaison = Inv($"{nom} : .NET={mesure.ToString(format, CultureInfo.InvariantCulture)} ; Python cité={citation.ToString(format, CultureInfo.InvariantCulture)} ; Δ={delta:F6} ; seuil={seuil:F6}");
    Exiger(delta <= seuil, comparaison + " — ÉCART");
    Console.WriteLine(comparaison + " — CONFORME");
}

Console.WriteLine();
Console.WriteLine("Attestation de parité par section");
AttesterLot("Référence", instancesRef);
AttesterLot("Extension", instancesNew);
AttesterLot("Global", allInst);
ComparerSynthese("Écart min", bilanEcarts.Min, syntheseEcartsPython.Min, seuilExtremumEcart, "F4");
ComparerSynthese("Écart max", bilanEcarts.Max, syntheseEcartsPython.Max, seuilExtremumEcart, "F4");
ComparerSynthese("Écart mean", bilanEcarts.Moyenne, syntheseEcartsPython.Moyenne, seuilMoyenneEcart, "F4");
ComparerSynthese("Rapport min", bilanRapports.Min, syntheseRapportsPython.Min, seuilExtremumRapport, "F3");
ComparerSynthese("Rapport max", bilanRapports.Max, syntheseRapportsPython.Max, seuilExtremumRapport, "F3");
ComparerSynthese("Rapport mean", bilanRapports.Moyenne, syntheseRapportsPython.Moyenne, seuilMoyenneRapport, "F3");
ComparerSynthese("Pourcentage de la borne atteint au pire", pourcentagePire, pourcentagePirePython, seuilPourcentagePire, "F0");

=== Cellule 3 : Confrontation Hoeffding sur 20 instances ===


Ecarts observes : min=0.0003, max=0.0108, mean=0.0056


Rapport ecart/borne : min=0.016, max=0.626, mean=0.296


--> 20/20 instances sous la borne Hoeffding (100 %)


Marge : le pire ecart atteint 63 % de la borne.


Pire rapport .NET : M=12, t=5, p=0.440, écart=0.0085, borne=0.0136.


... (7 outputs left out)

Rapport min : .NET=0.016 ; Python cité=0.016 ; Δ=0.000465 ; seuil=1.565078 — CONFORME


Rapport max : .NET=0.626 ; Python cité=0.698 ; Δ=0.072126 ; seuil=1.565078 — CONFORME


Rapport mean : .NET=0.296 ; Python cité=0.265 ; Δ=0.031235 ; seuil=1.422418 — CONFORME


Pourcentage de la borne atteint au pire : .NET=63 ; Python cité=70 ; Δ=7.412632 ; seuil=156.957809 — CONFORME


## Interprétation

**Résultat** : les 20 instances (12 de référence et 8 d'extension) restent sous la borne de Hoeffding avec `eps = 0.05`. Dans la sortie .NET, le rapport `ecart/borne` va de `0.016` à `0.626`, avec une moyenne de `0.296` : en moyenne, l'écart observé représente un peu moins d'un tiers de sa borne.

**Pourquoi ce carnet précède la capstone MIMO-04** : la sous-série MIMO-01/02/03/04 utilise un invariant de comptage sur l'ensemble des flips détectés. Lean-19 vérifie empiriquement cet invariant dans le modèle de Bernoulli et sert d'étalon : une déviation systématique des résultats de la capstone par rapport à ce banc d'essai signale soit un bug, soit un phénomène à documenter.

**Limites** : (a) le modèle est un BSC simple, pas le canal MIMO général ; (b) la borne de Hoeffding est une borne de concentration, pas un estimateur exact ; (c) le rapport moyen est empirique, pas une borne inférieure. Un autre carnet pourrait dériver une borne plus serrée, par exemple avec Chernoff.

La synthèse affiche `--> 20/20 instances sous la borne Hoeffding (100 %)`. Le pire rapport est observé pour `M=12, t=5, p=0.440` : l'écart de `0.0085` atteint `63 %` de la borne de `0.0136`, sans la saturer.

## Exercices

### Exercice 1 : calculer l'index sur une instance donnee

Implementer `p_hat_instance(M, t, p, n_runs, seed=0)` qui retourne `(p_hat, ecart, borne, tient)` pour une instance unique. Verifier que l'invariant tient pour `M=10, t=5, p=0.3, n_runs=5000`.

Les trois exercices prolongent la mécanique des cellules 1 à 3 : calcul de l'index sur une instance unique, énoncé de l'inégalité de Hoeffding, extension au cas multi-passe. Compétence visée : passer d'une boucle figée sur 12 instances à une fonction réutilisable sur n'importe quelle instance.

In [5]:
// Illustration 1 — Exercice : p_hat_instance
(double p_hat, double ecart, double borne, bool tient)? p_hat_instance(int M, int t, double p, int n_runs, int seed = 0)
{
    // TODO etudiant : retourner (p_hat, ecart, borne, tient)
    return null;
}

// Test : M=10, t=5, p=0.3, n_runs=5000
var resultatInstance = p_hat_instance(10, 5, 0.3, 5000);
if (!resultatInstance.HasValue)
    Console.WriteLine("Exercice 1 a completer");

Exercice 1 a completer


### Exercice 2 : preuve de l'inegalite de Hoeffding

Citer (sans la redemontrer) l'inegalite de Hoeffding pour une somme de variables aleatoires iid bornees dans `[0, 1]`. Quel est le lien avec la borne utilisee dans ce carnet ? Pourquoi `eps = 0.05` plutot que `eps = 0.01` ?

In [6]:
// Illustration 2 — Exercice : hoeffding_inegalite
(double borne, string justification)? hoeffding_inegalite(int N, double eps = 0.05)
{
    // TODO etudiant : retourner la borne et une breve justification (1-2 phrases)
    return null; // null -> TODO etudiant
}

var reponseHoeffding = hoeffding_inegalite(10000);
if (!reponseHoeffding.HasValue)
    Console.WriteLine("Exercice 2 a completer");

Exercice 2 a completer


### Exercice 3 : etendre au cas multi-passe (3+ flips par pas)

Modifier le modele BSC pour permettre jusqu'a `k = 3` flips par pas (avec probabilite `p_k` pour chaque nombre de flips). Verifier que l'invariant tient pour `k = 2` et `k = 3` sur 5 instances.

In [7]:
// Illustration 3 — Exercice : p_hat_multipasse
double? p_hat_multipasse(int M, int t, double[] p_k, int k_max, int n_runs, int seed = 0)
{
    // TODO etudiant : modele multi-passe (1, 2 ou k_max flips par pas)
    return null;
}

// Test : k=2, p_1=0.3, p_2=0.1, n_runs=3000
var resultatMultipasse = p_hat_multipasse(8, 4, new[] { 0.3, 0.1 }, 2, 3000);
if (!resultatMultipasse.HasValue)
    Console.WriteLine("Exercice 3 a completer");

Exercice 3 a completer


## Conclusion et references

Lean-19 etablit que la detection MIMO par comptage de flips est empiriquement verifiee sur 20 instances aleatoires. Ce carnet precede la sous-serie MIMO-01..04 et fournit un etalon pour la verification independante des resultats de la capstone.

**References** :

- `Lean-17-Conway-Bridge-to-Knots-Python.ipynb` -- le carnet precedent, qui pose le pont Conway (machine de Turing jouet) vers la theorie des nœuds.
- `Lean-20-Capstone-Digestions-Tao-Python.ipynb` -- la capstone de la serie, qui digere les techniques de Terence Tao sur les flots de Ricci.
- [MIMO-01 -- Descente Budget](MIMO/MIMO-01-Descente-Budget-Lean-Python.ipynb) -- le carnet qui pose la borne de budget pour la detection MIMO.
- [MIMO/README.md](MIMO/README.md) -- la descente MIMO-01..04, qui consomme Lean-19 comme etalon.

L'étalon chiffré du port .NET est de `20/20` instances sous la borne, avec un rapport `ecart/borne` maximal de `0.626` et moyen de `0.296`. Le carnet Python affiche respectivement `0.698` et `0.265` pour ces deux rapports : ce sont des références citées, pas des résultats .NET. Les chiffres du port donnent aux carnets MIMO-01..04 un point de comparaison explicite.

L'« Attestation de parité par section » donne, pour la référence, l'extension et le bilan global, l'écart maximal sur les bornes affichées et l'écart maximal entre les mesures .NET et les références Python citées. Elle compare aussi les statistiques de synthèse à leurs seuils respectifs.

> Source du banc d'essai : [Version Python](Lean-19-Detection-MIMO-Index-Python.ipynb).